# OLS Regression — Week 2
**Aayan Mulani │ Decimal Point Analytics Preparation**

OLS (Ordinary Least Squares) regression finds the linear relationship between two variables
by minimising the sum of squared errors. In finance, regression is used to model how one
asset moves relative to another — for example, how Nifty returns respond to Sensex returns.
This notebook builds a simple OLS model using the two major Indian indices.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

# Download Nifty 50 (Y) and Sensex (X) — 5 years of daily data
# .squeeze() converts single-column DataFrame → Series (required for newer yfinance)
nifty  = yf.download('^NSEI',  start='2020-01-01', end='2025-01-01')['Close'].squeeze()
sensex = yf.download('^BSESN', start='2020-01-01', end='2025-01-01')['Close'].squeeze()

# Combine into one DataFrame and compute daily returns
df = pd.DataFrame({'Nifty': nifty, 'Sensex': sensex}).dropna()
df['Nifty_Ret']  = df['Nifty'].pct_change()
df['Sensex_Ret'] = df['Sensex'].pct_change()
df.dropna(inplace=True)

print(df.shape)
print(df.head())

## 1. Scatter Plot — Visualising the Relationship

Before running any regression, always plot X vs Y. If the scatter plot shows no pattern,
regression won't help. If it shows a clear linear cloud, OLS is appropriate.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df['Sensex_Ret'], df['Nifty_Ret'], alpha=0.3, color='steelblue', s=10)
plt.xlabel('Sensex Daily Return')
plt.ylabel('Nifty Daily Return')
plt.title('Nifty vs Sensex — Daily Returns (2020–2025)', fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. OLS Regression — Nifty Returns ~ Sensex Returns

Model: Nifty_Return = b0 (intercept) + b1 × Sensex_Return + error

statsmodels requires us to manually add the intercept column using sm.add_constant().
Without it, the model forces the regression line through the origin (b0 = 0),
which is usually incorrect.

In [ ]:
# Prepare X and Y
X = sm.add_constant(df['Sensex_Ret'])  # adds a column of 1s for the intercept
Y = df['Nifty_Ret']

# Fit the OLS model
model = sm.OLS(Y, X).fit()

# Print the full regression summary
print(model.summary())

## Interpretation of OLS Results

**R² = 0.993:** The Sensex explains 99.3% of Nifty's daily return variation.
This is exceptionally high and expected — both indices track India's large-cap
equity universe and share most of the same underlying stocks.

**Slope (b1) = 0.9823, p = 0.000:** For every 1% move in the Sensex, the Nifty
moves approximately 0.9823% in the same direction. This coefficient is highly
statistically significant (p essentially zero). The Nifty is slightly less
volatile than the Sensex on a daily basis.

**Intercept (b0) = 0.0000288, p = 0.317:** Not statistically significant.
Nifty has no independent daily drift when Sensex is flat. We cannot reject the
null hypothesis that b0 = 0.

**Durbin-Watson = 2.034:** Close to 2.0 — no autocorrelation in residuals. The
errors are not serially correlated, which satisfies one of OLS's key assumptions.

**Jarque-Bera p = 0.00, Kurtosis = 11.98:** Residuals are not normally distributed
— fat tails remain even after accounting for Sensex. This is typical of financial
return data and a known limitation of standard OLS in markets.

## 3. Regression Line — Best Fit Visualised

Plotting the OLS regression line over the scatter confirms visually what the
summary table told us numerically — a near-perfect linear fit between Sensex
and Nifty daily returns.

In [ ]:
# Sort values so the regression line draws cleanly
plot_df = df[['Sensex_Ret', 'Nifty_Ret']].copy()
plot_df['Fitted'] = model.fittedvalues
plot_df = plot_df.sort_values('Sensex_Ret')

plt.figure(figsize=(8, 6))
plt.scatter(df['Sensex_Ret'], df['Nifty_Ret'],
            alpha=0.3, color='steelblue', s=10, label='Actual returns')
plt.plot(plot_df['Sensex_Ret'], plot_df['Fitted'],
         color='red', linewidth=2, label='OLS Regression Line')

b0 = model.params['const']
b1 = model.params['Sensex_Ret']
plt.title(f'OLS Fit: Nifty = {b0:.5f} + {b1:.4f} × Sensex  |  R² = 0.993',
          fontsize=12, fontweight='bold')
plt.xlabel('Sensex Daily Return')
plt.ylabel('Nifty Daily Return')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Regression Diagnostics — Testing OLS Assumptions

A high R² does not guarantee a valid model. OLS produces reliable coefficients
only when 5 key assumptions hold. We test each one below using residual plots,
the Durbin-Watson statistic, a Q-Q plot, and VIF.

In [ ]:
# Extract residuals and fitted values from the model
residuals = model.resid
fitted    = model.fittedvalues

print(f'Residuals — Mean: {residuals.mean():.6f}')
print(f'Residuals — Std:  {residuals.std():.6f}')
print(f'Min residual: {residuals.min():.4f}')
print(f'Max residual: {residuals.max():.4f}')

## 4.1 Assumption 1 & 2 — Linearity and Homoscedasticity

Plot residuals vs fitted values. A good model shows a random horizontal band
centred at zero — no pattern, no funnel shape. Any systematic pattern means
the linearity or constant-variance assumption is violated.

In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(fitted, residuals, alpha=0.3, color='steelblue', s=10)
plt.axhline(0, color='red', linewidth=1.5, linestyle='--')
plt.xlabel('Fitted Values (Predicted Nifty Return)')
plt.ylabel('Residuals')
plt.title('Residuals vs Fitted Values — Homoscedasticity Check',
          fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4.2 Assumption 3 — Normality of Residuals (Q-Q Plot)

A Q-Q (Quantile-Quantile) plot compares the distribution of our residuals
against a perfect normal distribution. If residuals are normal, all points
fall on the diagonal reference line. Deviations at the ends (tails) indicate
fat tails — more extreme errors than a normal distribution would predict.

In [ ]:
fig = sm.qqplot(residuals, line='s', alpha=0.3)
plt.title('Q-Q Plot — Normality of Residuals', fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4.3 Assumption 4 — No Autocorrelation (Residuals Over Time)

Residuals should be random over time — knowing today's error should not help
predict tomorrow's. We already saw Durbin-Watson = 2.034 in the model summary
(close to 2.0 = no autocorrelation). Here we visualise residuals over time to
confirm there is no systematic pattern or clustering.

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(residuals.index, residuals, color='steelblue', linewidth=0.7, alpha=0.8)
plt.axhline(0, color='red', linewidth=1.2, linestyle='--')
plt.title('Residuals Over Time — Autocorrelation Check',
          fontsize=13, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Residual')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4.4 Assumption 5 — No Multicollinearity (VIF)

VIF (Variance Inflation Factor) measures how much the variance of a coefficient
increases because of correlation with other predictors. With only one predictor
(Sensex), multicollinearity cannot exist by definition — but we compute VIF
anyway to practice the workflow used in multiple regression.

Rule of thumb: VIF < 5 = acceptable, VIF > 10 = serious problem.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Build a DataFrame of VIF scores for each predictor
vif_data = pd.DataFrame()
vif_data['Variable'] = ['const', 'Sensex_Ret']
vif_data['VIF'] = [variance_inflation_factor(model.model.exog, i)
                   for i in range(model.model.exog.shape[1])]

print(vif_data)

## 4.5 Diagnostics Summary — All 5 OLS Assumptions

| Assumption | Test Used | Result | Verdict |
|---|---|---|---|
| 1. Linearity | Scatter plot (X vs Y) | Clear linear cloud | ✅ Satisfied |
| 2. Homoscedasticity | Residuals vs Fitted plot | Mild widening at extremes | ⚠️ Minor violation |
| 3. Normality of Residuals | Q-Q Plot + Jarque-Bera | Fat tails, S-curve deviation | ❌ Violated |
| 4. No Autocorrelation | Residuals over time + Durbin-Watson = 2.034 | Random, no pattern | ✅ Satisfied |
| 5. No Multicollinearity | VIF = 1.0 | No correlation between predictors | ✅ Satisfied |

**Conclusion:** The model is strong (R² = 0.993, slope highly significant) but the
residuals show fat tails and slight heteroscedasticity at extremes — both common
in financial return data. For production use, robust standard errors or a
GARCH model would better handle the non-constant variance in volatility.